# BB84 Protocol with Noiseless Channel

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

- Alice's state: array of 0s and 1s denoting the bit values to be encoded
- Alice's bases: array of 0s and 1s denoting the basis used for encoding
  - 0 -> Computational basis
  - 1 -> Hadamard basis
- Bob's bases: array of 0s and 1s denoting the basis used for measurement
  - 0 -> Computational basis
  - 1 -> Hadamard basis

In [ ]:
num_qubits = 16

## Randomly choose Alice's encoding basis, Alice's bit values, and Bob's measurement basis

In [ ]:
alice_basis = np.random.randint(2, size=num_qubits)
alice_state = np.random.randint(2, size=num_qubits)
bob_basis = np.random.randint(2, size=num_qubits)

In [ ]:
# Print the randomly generated data
print(f"Alice's State:\t {np.array2string(alice_state)}")
print(f"Alice's Bases:\t {np.array2string(alice_basis)}")
print(f"Bob's Bases:\t {np.array2string(bob_basis)}")

## Create a quantum circuit with num_qubits qubits

In [ ]:
circuit = QuantumCircuit(num_qubits)

circuit.barrier()

### Alice prepares each qubit according to her chosen bit value and basis

In [ ]:
for i in range(len(alice_basis)):
    # If Alice wants to encode bit 1, apply X to flip |0> -> |1>
    if alice_state[i] == 1:
        circuit.x(i)

    # If Alice uses the Hadamard basis, apply H
    # This maps computational basis states into the diagonal basis
    if alice_basis[i] == 1:
        circuit.h(i)

circuit.barrier()

### Bob chooses his measurement basis
If Bob wants to measure in the Hadamard basis, he applies H before the final computational-basis measurement

In [ ]:
for i in range(len(bob_basis)):
    if bob_basis[i] == 1:
        circuit.h(i)

### Measure all qubits

In [ ]:
circuit.measure_all()

In [ ]:
# Draw the circuit
circuit.draw('mpl')

## Use AerSimulator to simulate the quantum circuit

In [ ]:
simulator = AerSimulator()

## Run the circuit only once (one BB84 transmission round)

In [ ]:
job = simulator.run(circuit, shots=1)

In [ ]:
# Collect the measurement result
result = job.result()
counts = result.get_counts()

# Since shots=1, there is only one observed bitstring
key = counts.most_frequent()

## Build the sifted key:
keep only the bits where Alice and Bob happened to choose the same basis

In [ ]:
encryption_key = ''
for i in range(num_qubits):
    if alice_basis[i] == bob_basis[i]:
        encryption_key += str(key[i])

print(f"Key: {encryption_key}")